In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import string
import nltk
import re
import string
from nltk.corpus import stopwords
import pkg_resources
import pickle
import nltk
import re, string, json
from tqdm.notebook import tqdm
from nltk.tokenize import word_tokenize

C:\Users\dell\AppData\Local\Temp\ipykernel_9444\2817552085.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
from numpy import array
from pickle import dump
from keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Embedding

from random import randint
from pickle import load
from keras.models import load_model
from keras.preprocessing.sequence import pad_sequences

In [3]:
# load doc into memory
def load_doc(filename):
    # open the file as read only
    file = open(filename, 'r')
    # read all text
    text = file.read()
    # close the file
    file.close()
    return text

# turn a doc into clean tokens
def clean_doc(doc):
    # replace '--' with a space ' '
    doc = doc.replace('--', ' ')
    # split into tokens by white space
    tokens = doc.split()
    # remove punctuation from each token
    table = str.maketrans('', '', string.punctuation)
    tokens = [w.translate(table) for w in tokens]
    # remove remaining tokens that are not alphabetic
    tokens = [word for word in tokens if word.isalpha()]
    # make lower case
    tokens = [word.lower() for word in tokens]
    return tokens

# save tokens to file, one dialog per line
def save_doc(lines, filename):
    data = '\n'.join(lines)
    file = open(filename, 'w')
    file.write(data)
    file.close()

In [4]:
def tokenize_twitter(sentences):
    """
    Tokenize sentences into tokens (words)
    
    Args:
        sentences: List of strings
    
    Returns:
        List of lists of tokens
    """
    print("Starting Cleaning Process")
    tokenized_sentences = []
    for sentence in tqdm(sentences):
        
        # Convert to lowercase letters
        sentence = cleanhtml(sentence)
        sentence = _replace_urls(sentence)
        sentence = remove_email(sentence)
        sentence = re.sub(r'[^a-zA-Z]', ' ', sentence)
        sentence = sentence.lower()
        sentence = misc(sentence)
        

        # tokenized = nltk.word_tokenize(sentence)
        
        # append the list of words to the list of lists
        # tokenized_sentences.append(tokenized)
        tokenized_sentences.append(sentence)
    
    return tokenized_sentences

def cleanhtml(raw_html):
    cleanr = re.compile('<.*?>')
    cleantext = re.sub(cleanr, '', raw_html)
    return cleantext


def _replace_urls(data):
    #Removing URLs with a regular expression
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    data = url_pattern.sub(r'', data)
    return data

def remove_email(data):
    # Remove Emails
    data = re.sub('\S*@\S*\s?', '', data)
    return data

def misc(data):
    # Remove new line characters
    data = re.sub('\s+', ' ', data)
    # Remove distracting single quotes
    data = re.sub("\'", "", data)
    data = re.sub("ww+", "", data)
    # Removing roman-case:
    MAYBE_ROMAN = re.compile(r'(\b[MDCLXVI]+\b)(\.)?', re.I)
    data = re.sub(MAYBE_ROMAN, "", data)
    return data

In [5]:
def littleCleaning(sentences):
    print("Starting cleaning Process")
    ret_list = []
    for sentence in sentences:
      words = sentence.split(" ")
      if len(words) > 5:
        ret_list.append(sentence)
      else:
        continue
    return ret_list

In [13]:
sample = """We the People of the United States, in Order to form a more perfect Union,
establish Justice, insure domestic Tranquility, provide for the common defence,
promote the general Welfare, and secure the Blessings of Liberty to ourselves and our Posterity,
do ordain and establish this Constitution for the United States of America."""
with open(r'd:/project_learning/Nlp-2/predict_next_word/republic.txt', 'w', encoding='utf-8') as f:
    f.write(sample)


In [14]:
with open(r'd:/project_learning/Nlp-2/predict_next_word/republic.txt', encoding='utf-8') as f:
    text = f.read().lower()
print('length of the corpus is:', len(text))


length of the corpus is: 327


In [15]:
# Converting the data into lists.

data_list = text.split(".")
data_list[:20]

['we the people of the united states, in order to form a more perfect union,\nestablish justice, insure domestic tranquility, provide for the common defence,\npromote the general welfare, and secure the blessings of liberty to ourselves and our posterity,\ndo ordain and establish this constitution for the united states of america',
 '']

In [16]:
pro_sentences = []

def normalization_pipeline(sentences):
    print("Starting Normalization Process")
    sentences = tokenize_twitter(sentences)
    sentences = littleCleaning(sentences)
    print("Normalization Process Finished")
    return sentences

pro_sentences = normalization_pipeline(data_list)
pro_sentences[: 5]

Starting Normalization Process
Starting Cleaning Process


  0%|          | 0/2 [00:00<?, ?it/s]

Starting cleaning Process
Normalization Process Finished


['we the people of the united states in order to form a more perfect union establish justice insure domestic tranquility provide for the common defence promote the general welfare and secure the blessings of liberty to ourselves and our posterity do ordain and establish this constitution for the united states of america']

In [17]:
len(pro_sentences)

1

In [18]:
# Structuring th etext into a paragraph:

dataText = "".join(pro_sentences[: 700])
dataText[: 200]

'we the people of the united states in order to form a more perfect union establish justice insure domestic tranquility provide for the common defence promote the general welfare and secure the blessin'

In [19]:
# turn a doc into clean tokens
def clean_doc(doc):
    # replace '--' with a space ' '
    doc = doc.replace('--', ' ')
    # split into tokens by white space
    tokens = doc.split()
    # remove punctuation from each token
    table = str.maketrans('', '', string.punctuation)
    tokens = [w.translate(table) for w in tokens]
    # remove remaining tokens that are not alphabetic
    tokens = [word for word in tokens if word.isalpha()]
    # make lower case
    tokens = [word.lower() for word in tokens]
    return tokens

In [20]:
# clean document
tokens = clean_doc(dataText)
print(tokens[:200])
print('Total Tokens: %d' % len(tokens))
print('Unique Tokens: %d' % len(set(tokens)))

['we', 'the', 'people', 'of', 'the', 'united', 'states', 'in', 'order', 'to', 'form', 'a', 'more', 'perfect', 'union', 'establish', 'justice', 'insure', 'domestic', 'tranquility', 'provide', 'for', 'the', 'common', 'defence', 'promote', 'the', 'general', 'welfare', 'and', 'secure', 'the', 'blessings', 'of', 'liberty', 'to', 'ourselves', 'and', 'our', 'posterity', 'do', 'ordain', 'and', 'establish', 'this', 'constitution', 'for', 'the', 'united', 'states', 'of', 'america']
Total Tokens: 52
Unique Tokens: 38


In [21]:
# organize into sequences of tokens
length = 50 + 1
sequences = list()
for i in range(length, len(tokens)):
    # select sequence of tokens
    seq = tokens[i-length:i]
    # convert into a line
    line = ' '.join(seq)
    # store
    sequences.append(line)
print('Total Sequences: %d' % len(sequences))

Total Sequences: 1


In [25]:
# save tokens to file, one dialog per line
def save_doc(lines, filename):
    data = "\n".join(lines)
    with open(filename, "w", encoding="utf-8") as f:
        f.write(data)

# Use an absolute Windows path that points to the notebook’s folder
out_filename = r"d:\project_learning\Nlp-2\predict_next_word\republic_sequences.txt"
save_doc(sequences, out_filename)
print(f"Saved {len(sequences)} sequences to {out_filename}")


Saved 1 sequences to d:\project_learning\Nlp-2\predict_next_word\republic_sequences.txt


In [26]:
in_filename = r"d:\project_learning\Nlp-2\predict_next_word\republic_sequences.txt"
doc = load_doc(in_filename)          # returns a single string with newline separators
lines = doc.split('\n')               # list of tokenised sentences (one per line)

tokenizer = Tokenizer()
tokenizer.fit_on_texts(lines)        # build the vocabulary
sequences = tokenizer.texts_to_sequences(lines)

vocab_size = len(tokenizer.word_index) + 1

sequences = array(sequences)          # convert list‑of‑lists → NumPy array
X, y = sequences[:, :-1], sequences[:, -1]   # all but last token → input, last token → target
y = to_categorical(y, num_classes=vocab_size)

# Length of each input sequence (used later for padding / model input shape)
seq_length = X.shape[1]
print(f"Loaded {len(lines)} sequences")
print(f"Vocabulary size: {vocab_size}")
print(f"Sequence length (input): {seq_length}")


Loaded 1 sequences
Vocabulary size: 38
Sequence length (input): 50


In [27]:
# define model
model = Sequential()
model.add(Embedding(vocab_size, 50, input_length=seq_length))
model.add(LSTM(50, return_sequences=True))
model.add(LSTM(50))
model.add(Dense(50, activation='relu'))
model.add(Dense(vocab_size, activation='softmax'))
print(model.summary())
# compile model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
# fit model
batch_size=128
epochs=50
model.fit(X, y, batch_size=batch_size, epochs=epochs)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 50, 50)            1900      
                                                                 
 lstm (LSTM)                 (None, 50, 50)            20200     
                                                                 
 lstm_1 (LSTM)               (None, 50)                20200     
                                                                 
 dense (Dense)               (None, 50)                2550      
                                                                 
 dense_1 (Dense)             (None, 38)                1938      
                                                                 
Total params: 46,788
Trainable params: 46,788
Non-trainable params: 0
_________________________________________________________________
None
Epoch 1/50
1/1 [==============================]

In [28]:

model_path = r"d:\project_learning\Nlp-2\predict_next_word\DataScience-Pianalytix-Models\nexWordPredict\nextWord.h5"
model.save(model_path)
print(f"Model saved to {model_path}")

tokenizer_path = r"d:\project_learning\Nlp-2\predict_next_word\DataScience-Pianalytix-Models\tokenizer.pkl"
with open(tokenizer_path, "wb") as f:
    dump(tokenizer, f)

print(f"Tokenizer saved to {tokenizer_path}")


Model saved to d:\project_learning\Nlp-2\predict_next_word\DataScience-Pianalytix-Models\nexWordPredict\nextWord.h5
Tokenizer saved to d:\project_learning\Nlp-2\predict_next_word\DataScience-Pianalytix-Models\tokenizer.pkl


In [29]:
# generate a sequence from a language model
import numpy as np

def generate_seq(model, tokenizer, seq_length, seed_text, n_words):
    result = list()
    in_text = seed_text
    # generate a fixed number of words
    for _ in range(n_words):
        # encode the text as integer
        encoded = tokenizer.texts_to_sequences([in_text])[0]
        # truncate sequences to a fixed length
        encoded = pad_sequences([encoded], maxlen=seq_length, truncating='pre')
        # predict probabilities for each word
        # yhat = model.predict_classes(encoded, verbose=0)
        predict_x=model.predict(encoded) 
        yhat=np.argmax(predict_x,axis=1)
        # map predicted word index to word
        out_word = ''
        for word, index in tokenizer.word_index.items():
            if index == yhat:
                out_word = word
                break
        # append to input
        in_text += ' ' + out_word
        result.append(out_word)
    return ' '.join(result)

In [30]:

in_filename = r"d:\project_learning\Nlp-2\predict_next_word\republic_sequences.txt"

doc = load_doc(in_filename)          # ← reads the file
lines = doc.split('\n')               # each line = one tokenised sentence

seq_length = len(lines[0].split()) - 1   # we later predict the last token

# Quick sanity check
print(f"Number of sequences   : {len(lines)}")
print(f"First sequence (raw)   : {lines[0]}")
print(f"Input length per sample: {seq_length}")


Number of sequences   : 1
First sequence (raw)   : we the people of the united states in order to form a more perfect union establish justice insure domestic tranquility provide for the common defence promote the general welfare and secure the blessings of liberty to ourselves and our posterity do ordain and establish this constitution for the united states of
Input length per sample: 50


In [32]:

from keras.models import load_model          # keras‑tensorflow import
from pickle import load                     # un‑pickle the tokenizer
from random import randint                 # for picking a random seed text

model_path     = r"d:\project_learning\Nlp-2\predict_next_word\DataScience-Pianalytix-Models\nexWordPredict\nextWord.h5"
tokenizer_path = r"d:\project_learning\Nlp-2\predict_next_word\DataScience-Pianalytix-Models\tokenizer.pkl"

# Load the model
model = load_model(model_path)

# Load the tokenizer
tokenizer = load(open(tokenizer_path, "rb"))

seed_idx  = randint(0, len(lines) - 1)          # pick a valid index
seed_text = lines[seed_idx]

print("\n🔸 Seed text:\n" + seed_text + "\n")

# Generate 12 new words using the helper you defined earlier
generated = generate_seq(model, tokenizer, seq_length, seed_text, 12)

print("🔹 Generated continuation:\n" + generated)



🔸 Seed text:
we the people of the united states in order to form a more perfect union establish justice insure domestic tranquility provide for the common defence promote the general welfare and secure the blessings of liberty to ourselves and our posterity do ordain and establish this constitution for the united states of

🔹 Generated continuation:
of of of of of of of of of of of of
